# AI Lab AL2002 - Final Cheat Sheet

**Purpose:** A complete last-day revision and coding template for solving any AL2002 lab exam question.
Copy the relevant section, swap in your data/variables, and run.

---

## Overall Summary

| Topic | One-Line Summary |
|---|---|
| **CSP (OR-Tools)** | Define variables + domains, add constraints, call solver |
| **Probability** | P(A) = favorable / total; Bayes: P(A|B) = P(B|A)*P(A)/P(B) |
| **Minimax** | MAX player picks max child; MIN player picks min child |
| **Alpha-Beta** | Minimax with pruning; skip branches that cannot affect result |
| **EDA** | Load -> inspect -> handle nulls -> encode -> scale |
| **Linear Regression** | Fit line y = mx + b; minimize MSE |
| **Decision Tree** | Split on best Information Gain (entropy reduction) |
| **KNN** | Classify by majority vote of K nearest neighbors |
| **K-Means** | Assign points to nearest centroid; update centroid; repeat |

---

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, confusion_matrix
)

from ortools.sat.python import cp_model

print("All imports successful.")

All imports successful.


---
## 2. Data Handling and EDA

**Steps:**
1. Load dataset
2. Inspect shape, types, missing values
3. Handle missing values (drop or fill)
4. Encode categorical columns
5. Scale features (StandardScaler or MinMaxScaler)

In [ ]:
# --- LOAD ---
df = pd.read_csv("your_file.csv")       # <-- replace with your file

# --- INSPECT ---
print(df.shape)                         # rows, columns
print(df.dtypes)                        # column types
df.head()                               # first 5 rows
df.info()                               # non-null counts
df.describe()                           # stats summary

# --- MISSING VALUES ---
print(df.isnull().sum())               # count nulls per column

df.dropna(inplace=True)                # option 1: drop rows with nulls
# df.fillna(df.mean(), inplace=True)   # option 2: fill with column mean
# df['col'].fillna('Unknown', inplace=True)  # option 3: fill with a value

# --- ENCODE CATEGORICAL ---
le = LabelEncoder()
# df['categorical_col'] = le.fit_transform(df['categorical_col'])

# One-hot encoding alternative:
# df = pd.get_dummies(df, columns=['categorical_col'])

# --- SCALE FEATURES ---
scaler = StandardScaler()              # mean=0, std=1
# scaler = MinMaxScaler()             # range [0, 1]

# X = df.drop('target_col', axis=1)
# y = df['target_col']
# X_scaled = scaler.fit_transform(X)

print("EDA template ready.")

---
## 3. Linear Regression

**Steps:**
1. Define X (features) and y (target)
2. Split into train/test
3. Fit the model
4. Predict on test set
5. Evaluate with MSE and R2

**Key formulas:**
- Prediction: `y = w0 + w1*x1 + w2*x2 + ...`
- MSE = mean((y_true - y_pred)^2)
- R2 = 1 - (SS_res / SS_tot)  -- closer to 1 is better

In [ ]:
# --- DATA SETUP ---
# Replace with your actual DataFrame
# X = df[['feature1', 'feature2']]   # feature columns
# y = df['target']                    # target column

# Dummy data for demonstration
X = pd.DataFrame({'feature1': [1, 2, 3, 4, 5],
                  'feature2': [2, 4, 5, 4, 5]})
y = pd.Series([1, 2, 3, 3, 5])

# --- SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- TRAIN ---
model = LinearRegression()
model.fit(X_train, y_train)

# --- PREDICT ---
y_pred = model.predict(X_test)

# --- EVALUATE ---
print("Coefficients:", model.coef_)
print("Intercept:   ", model.intercept_)
print("MSE:         ", mean_squared_error(y_test, y_pred))
print("R2 Score:    ", r2_score(y_test, y_pred))

---
## 4. Decision Trees

**Key Concepts:**

**Entropy** measures impurity of a node:
```
Entropy(S) = -sum( p_i * log2(p_i) )
```
- Pure node (all same class) -> Entropy = 0
- Equal split -> Entropy = 1

**Information Gain** = how much entropy drops after a split:
```
IG(S, A) = Entropy(S) - sum( |Sv|/|S| * Entropy(Sv) )
```
Pick the attribute with the **highest IG** as the split.

**Manual Solving Steps:**
1. Calculate Entropy of the full dataset
2. For each attribute, calculate weighted entropy after split
3. IG = parent entropy - weighted child entropy
4. Choose attribute with highest IG as root
5. Recurse on each branch until pure or no attributes left

In [ ]:
# --- MANUAL ENTROPY AND IG CALCULATION ---
def entropy(labels):
    """Calculate entropy of a list of class labels."""
    from collections import Counter
    counts = Counter(labels)
    total = len(labels)
    return -sum((c / total) * np.log2(c / total) for c in counts.values() if c > 0)

def information_gain(parent_labels, split_groups):
    """IG = entropy(parent) - weighted avg entropy of children."""
    total = len(parent_labels)
    weighted = sum((len(g) / total) * entropy(g) for g in split_groups)
    return entropy(parent_labels) - weighted

# Example
parent = ['Yes', 'No', 'Yes', 'Yes', 'No']
group1 = ['Yes', 'Yes']
group2 = ['No', 'Yes', 'No']
print("Parent Entropy:", round(entropy(parent), 4))
print("IG of split:   ", round(information_gain(parent, [group1, group2]), 4))

print()

# --- SKLEARN DECISION TREE ---
# Replace X, y with your data
X_cls = pd.DataFrame({'f1': [1, 2, 3, 4, 5, 6],
                      'f2': [2, 3, 1, 5, 4, 6]})
y_cls = ['A', 'A', 'B', 'B', 'A', 'B']

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42
)

dt = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
print("DT Accuracy:", accuracy_score(y_test, y_pred))
print("Feature Importances:", dt.feature_importances_)

---
## 5. K-Nearest Neighbors (KNN)

**Concept:**
- Store all training points
- For a new point, find K closest training points (by Euclidean distance)
- Predict the majority class among those K neighbors

**Choosing K:**
- Small K -> low bias, high variance (overfitting)
- Large K -> high bias, low variance (underfitting)
- Common rule: try `K = sqrt(n)` and tune

**Euclidean Distance:**
```
d = sqrt( sum( (x_i - y_i)^2 ) )
```
Always scale features before using KNN.

In [ ]:
# --- KNN TEMPLATE ---
# Replace with your X, y
X_cls = pd.DataFrame({'f1': [1, 2, 3, 4, 5, 6, 7, 8],
                      'f2': [1, 3, 2, 5, 4, 7, 6, 8]})
y_cls = ['A', 'A', 'A', 'B', 'B', 'B', 'A', 'B']

# Scale first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cls)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_cls, test_size=0.25, random_state=42
)

# Try different K values
for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    print(f"K={k}  Accuracy={acc:.2f}")

# --- MANUAL DISTANCE CALCULATION ---
def euclidean(a, b):
    return np.sqrt(sum((ai - bi)**2 for ai, bi in zip(a, b)))

point_new  = [3, 4]
point_train = [1, 2]
print("\nEuclidean distance:", round(euclidean(point_new, point_train), 4))

---
## 6. K-Means Clustering

**Steps:**
1. Choose number of clusters K
2. Randomly initialize K centroids
3. Assign each point to the nearest centroid
4. Recompute centroid = mean of all points in cluster
5. Repeat steps 3-4 until centroids stop moving

**Choosing K:** Use the Elbow Method - plot inertia vs K; pick the bend point.

In [ ]:
# --- K-MEANS TEMPLATE ---
# Replace with your data
np.random.seed(42)
X_clust = np.vstack([
    np.random.randn(30, 2) + [0, 0],
    np.random.randn(30, 2) + [5, 5],
    np.random.randn(30, 2) + [0, 5]
])

# --- ELBOW METHOD ---
inertias = []
K_range = range(1, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)

plt.figure(figsize=(5, 3))
plt.plot(K_range, inertias, marker='o')
plt.title('Elbow Method')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

# --- FIT WITH CHOSEN K ---
K = 3
km = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = km.fit_predict(X_clust)

print("Cluster centers:")
print(km.cluster_centers_)

# --- CLUSTER PLOT ---
plt.figure(figsize=(5, 4))
for c in range(K):
    pts = X_clust[labels == c]
    plt.scatter(pts[:, 0], pts[:, 1], label=f'Cluster {c}')
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            marker='X', s=150, c='black', label='Centroids')
plt.title('K-Means Clusters')
plt.legend()
plt.tight_layout()
plt.show()

---
## 7. Probability and Z-Score

**Basic Rules:**
```
P(A)          = favorable outcomes / total outcomes
P(A and B)    = P(A) * P(B)            [if independent]
P(A or B)     = P(A) + P(B) - P(A and B)
P(A | B)      = P(A and B) / P(B)      [conditional]
Bayes: P(A|B) = P(B|A) * P(A) / P(B)
```

**Z-Score (standardization):**
```
z = (x - mean) / std
```
Z tells you how many standard deviations x is from the mean.

In [ ]:
# --- BASIC PROBABILITY ---
def probability(favorable, total):
    return favorable / total

def conditional(p_a_and_b, p_b):
    """P(A | B)"""
    return p_a_and_b / p_b

def bayes(p_b_given_a, p_a, p_b):
    """P(A | B) via Bayes theorem"""
    return (p_b_given_a * p_a) / p_b

# Example
print("P(drawing ace):", round(probability(4, 52), 4))
print("Bayes example: ", round(bayes(0.8, 0.01, 0.1), 4))

# --- Z-SCORE ---
def z_score(x, mean, std):
    return (x - mean) / std

data = [10, 20, 30, 40, 50]
mean = np.mean(data)
std  = np.std(data)

print("\nData:", data)
print("Mean:", mean, "  Std:", round(std, 4))
print("Z-scores:", [round(z_score(x, mean, std), 3) for x in data])

# Numpy shortcut
z_np = (np.array(data) - mean) / std
print("Z (numpy):  ", np.round(z_np, 3))

---
## 8. Minimax and Alpha-Beta Pruning

**Game Tree:**
- Two players: MAX (wants to maximize score) and MIN (wants to minimize score)
- Alternate turns; leaf nodes have terminal scores

**Minimax:**
- At MAX node: return max of children values
- At MIN node: return min of children values
- Recurse down to leaves, then back up

**Alpha-Beta Pruning:**
- alpha = best score MAX can guarantee so far (starts -inf)
- beta  = best score MIN can guarantee so far (starts +inf)
- Prune at MIN node if node value <= alpha (MAX would never go here)
- Prune at MAX node if node value >= beta  (MIN would never allow this)
- Produces same result as minimax but faster

In [ ]:
# --- MINIMAX ---
def minimax(node, depth, is_max, tree):
    """
    tree: dict mapping node -> list of children values or child nodes
    Leaf node: not in tree (or empty children)
    """
    if node not in tree or len(tree[node]) == 0:
        return node  # leaf: node is the score itself

    children = tree[node]
    if is_max:
        return max(minimax(child, depth - 1, False, tree) for child in children)
    else:
        return min(minimax(child, depth - 1, True, tree) for child in children)


# --- ALPHA-BETA PRUNING ---
def alpha_beta(node, depth, alpha, beta, is_max, tree):
    if node not in tree or len(tree[node]) == 0:
        return node

    if is_max:
        value = float('-inf')
        for child in tree[node]:
            value = max(value, alpha_beta(child, depth-1, alpha, beta, False, tree))
            alpha = max(alpha, value)
            if alpha >= beta:
                break   # beta cutoff (prune)
        return value
    else:
        value = float('inf')
        for child in tree[node]:
            value = min(value, alpha_beta(child, depth-1, alpha, beta, True, tree))
            beta = min(beta, value)
            if alpha >= beta:
                break   # alpha cutoff (prune)
        return value


# --- EXAMPLE TREE ---
# Internal nodes map to children; leaf nodes are integers (scores)
game_tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': [3, 5],
    'E': [2, 9],
    'F': [0, 7],
    'G': [4, 1],
}

mm_result = minimax('A', depth=3, is_max=True, tree=game_tree)
ab_result = alpha_beta('A', depth=3, alpha=float('-inf'), beta=float('inf'),
                       is_max=True, tree=game_tree)

print("Minimax result:    ", mm_result)
print("Alpha-Beta result: ", ab_result)
print("Both should match:", mm_result == ab_result)

---
## 9. CSP Using OR-Tools

**Steps:**
1. Create a `CpModel`
2. Define variables with `model.NewIntVar(lb, ub, name)`
3. Add constraints with `model.Add(...)`
4. Create `CpSolver` and call `solver.Solve(model)`
5. If `OPTIMAL` or `FEASIBLE`, read values with `solver.Value(var)`

**Common Constraints:**
```
model.Add(x != y)               # not equal
model.Add(x + y <= 10)          # sum constraint
model.AddAllDifferent([x,y,z])  # all different
model.AddBoolOr([b1, b2])       # at least one true
```

In [ ]:
# --- CSP EXAMPLE: Simple Scheduling ---
# Three tasks (A, B, C) must be assigned to time slots 1-3.
# Each task gets a different slot.

model = cp_model.CpModel()

# Variables: each task is assigned a slot in [1, 3]
task_a = model.NewIntVar(1, 3, 'task_a')
task_b = model.NewIntVar(1, 3, 'task_b')
task_c = model.NewIntVar(1, 3, 'task_c')

# Constraint: all tasks in different slots
model.AddAllDifferent([task_a, task_b, task_c])

# Extra constraint example: task_a must finish before task_b
model.Add(task_a < task_b)

# Solve
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("Solution found:")
    print("  Task A -> Slot", solver.Value(task_a))
    print("  Task B -> Slot", solver.Value(task_b))
    print("  Task C -> Slot", solver.Value(task_c))
else:
    print("No solution found.")

# --- TEMPLATE: Map Coloring / Variable Naming ---
# regions = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V']
# colors = {r: model.NewIntVar(0, 2, r) for r in regions}  # 3 colors: 0,1,2
# model.Add(colors['WA'] != colors['NT'])
# model.Add(colors['WA'] != colors['SA'])
# ... add all adjacency constraints ...

---
## 10. Plotting Templates

**Types:**
- Scatter plot: visualize two features against each other
- Line plot: show trends over time or ordered data
- Bar plot: compare categories
- Histogram: distribution of one feature

In [ ]:
x = np.linspace(0, 10, 50)
y = 2 * x + np.random.randn(50)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# --- SCATTER PLOT ---
axes[0].scatter(x, y, color='black', s=15)
axes[0].set_title('Scatter Plot')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')

# --- LINE PLOT ---
axes[1].plot(x, y, color='black', linewidth=1)
axes[1].set_title('Line Plot')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')

# --- HISTOGRAM ---
axes[2].hist(y, bins=10, color='gray', edgecolor='black')
axes[2].set_title('Histogram')
axes[2].set_xlabel('Value')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# --- BAR PLOT TEMPLATE ---
# categories = ['A', 'B', 'C']
# values     = [10, 25, 15]
# plt.bar(categories, values, color='gray', edgecolor='black')
# plt.title('Bar Chart')
# plt.show()

---
## 11. Full Example Usage (End-to-End)

Three small self-contained examples using the Iris dataset:
- Regression (predict petal length)
- Classification (predict species with Decision Tree)
- Clustering (K-Means on petal dimensions)

Replace `load_iris()` with your own dataset during the exam.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_all = pd.DataFrame(iris.data, columns=iris.feature_names)
y_all = pd.Series(iris.target, name='species')

print("Dataset shape:", X_all.shape)
print(X_all.head(3))
print()

# ---- REGRESSION EXAMPLE ----
# Predict petal length (col 2) from sepal length (col 0)
X_reg = X_all[['sepal length (cm)']]
y_reg = X_all['petal length (cm)']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=0
)
reg = LinearRegression().fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)
print("[Regression]")
print("  MSE :", round(mean_squared_error(yr_test, yr_pred), 4))
print("  R2  :", round(r2_score(yr_test, yr_pred), 4))
print()

# ---- CLASSIFICATION EXAMPLE ----
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=0
)
clf = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=0)
clf.fit(Xc_train, yc_train)
yc_pred = clf.predict(Xc_test)
print("[Decision Tree Classification]")
print("  Accuracy:", round(accuracy_score(yc_test, yc_pred), 4))
print("  Confusion Matrix:")
print(confusion_matrix(yc_test, yc_pred))
print()

# ---- CLUSTERING EXAMPLE ----
X_clust2 = X_all[['petal length (cm)', 'petal width (cm)']].values
km2 = KMeans(n_clusters=3, random_state=0, n_init=10)
cluster_labels = km2.fit_predict(X_clust2)
print("[K-Means Clustering]")
print("  Cluster counts:", np.bincount(cluster_labels))

plt.figure(figsize=(5, 4))
for c in range(3):
    pts = X_clust2[cluster_labels == c]
    plt.scatter(pts[:, 0], pts[:, 1], label=f'Cluster {c}', s=20)
plt.scatter(km2.cluster_centers_[:, 0], km2.cluster_centers_[:, 1],
            marker='X', s=120, c='black', label='Centroids')
plt.xlabel('Petal Length')
plt.ylabel('Petal Width')
plt.title('K-Means on Iris')
plt.legend()
plt.tight_layout()
plt.show()

---
## Quick Reference Card

### Metric Formulas
```
MSE       = mean( (y_true - y_pred)^2 )
RMSE      = sqrt(MSE)
R2        = 1 - SS_res/SS_tot          (1 = perfect)
Accuracy  = correct / total
Precision = TP / (TP + FP)
Recall    = TP / (TP + FN)
F1        = 2 * Precision * Recall / (Precision + Recall)
```

### sklearn Cheat Sheet
```python
model.fit(X_train, y_train)     # train
model.predict(X_test)           # predict classes / values
model.predict_proba(X_test)     # predict probabilities (classifiers)
model.score(X_test, y_test)     # default metric (accuracy / R2)
```

### OR-Tools Status Codes
```
cp_model.OPTIMAL   -> solution found and proven optimal
cp_model.FEASIBLE  -> a valid solution found (not necessarily optimal)
cp_model.INFEASIBLE -> no solution exists with given constraints
```

---
*End of cheat sheet. Good luck.*

qwen2.5-coder:7b prompt before asking questions:

You are an AI Lab assistant for a FAST-NUCES AL2002 student.

Your role is to help solve AI lab problems exactly according to the student’s syllabus and learning style. You must prioritize clarity, step-by-step solutions, and exam-focused answers.

---

## CORE BEHAVIOR RULES

* Always give step-by-step solutions
* Keep explanations simple and direct
* Do NOT overcomplicate
* Do NOT use unnecessary advanced syntax
* Prefer clarity over optimization
* Use exam-style solving methods
* When solving numericals, show all steps clearly
* When giving code, ensure it is correct and minimal

---

## SYLLABUS COVERAGE

You must be able to handle:

1. Data Handling and EDA

* Loading datasets
* Data inspection (head, info, describe)
* Handling missing values
* Encoding categorical variables
* Feature scaling (StandardScaler, MinMaxScaler)

2. Machine Learning Models

* Linear Regression
* Decision Trees (entropy, information gain, manual solving)
* K-Nearest Neighbors (KNN)
* K-Means Clustering

3. Constraint Satisfaction Problems

* Problem modeling (variables, domains, constraints)
* Backtracking
* OR-Tools (cp_model basic usage)

4. Game Trees

* Minimax algorithm
* Alpha-beta pruning
* Step-by-step evaluation of trees

5. Probability and Statistics

* Basic probability problems
* Z-score calculation and interpretation

6. Visualization

* Scatter plots
* Line plots
* Cluster visualization
* Interpretation of graphs

---

## OUTPUT FORMAT RULES

If the user asks for THEORY:

* Give short, clear explanation
* Use bullet points if needed
* Avoid long paragraphs

If the user asks for NUMERICAL:

* Solve step-by-step
* Show formulas used
* Show intermediate calculations

If the user asks for CODE:

* Provide clean Python code
* Use standard libraries: numpy, pandas, sklearn, matplotlib, ortools
* Include minimal comments explaining steps
* Use placeholders where needed (e.g., "your_file.csv")

If the user asks for BOTH:

* First explain briefly
* Then give step-by-step solution
* Then provide code

---

## DECISION TREE RULE (IMPORTANT)

When solving Decision Tree numericals:

1. Calculate entropy
2. Calculate information gain for each attribute
3. Select best attribute
4. Repeat recursively
5. Clearly show all calculations

---

## MINIMAX RULE

When solving game trees:

* Draw or describe tree levels
* Show MAX and MIN decisions clearly
* Evaluate leaf nodes first
* Propagate values upward
* Clearly indicate pruned branches (for alpha-beta)

---

## EDA RULE

When working with datasets:

1. Show how to load data
2. Inspect dataset
3. Clean data
4. Encode categorical variables
5. Scale features

---

## IMPORTANT RESTRICTIONS

* Do NOT use complex libraries beyond syllabus
* Do NOT give unnecessary theory
* Do NOT skip steps
* Do NOT assume prior steps are known

---

## GOAL

Help the student solve lab exam questions quickly, correctly, and step-by-step using only the concepts they have been taught.

Act like a practical lab instructor, not a researcher.

Begin.


In [1]:
from ortools.sat.python import cp_model

def create_csp():
    # Create the model.
    model = cp_model.CpModel()

    # Define variables
    T1 = model.NewIntVar(0, 4, 'T1')
    T2 = model.NewIntVar(0, 4, 'T2')
    T3 = model.NewIntVar(0, 4, 'T3')
    T4 = model.NewIntVar(0, 4, 'T4')
    T5 = model.NewIntVar(0, 4, 'T5')

    # Constraints
    model.Add(T1 != T3)
    model.Add(T5 != T1)
    model.Add(T5 != T2)
    model.Add(T4 != T5)
    model.Add(T2 < T4)
    model.Add(T1 < T3)

    # Natural Language Processing must be in Slot 3
    model.Add((T3 == 0) | (T3 == 1))

    # Computer Vision cannot be in the final slot
    model.Add(T4 != 4)

    # Solve the model.
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL:
        schedule = {
            'T1': f"Slot {solver.Value(T1) + 1}",
            'T2': f"Slot {solver.Value(T2) + 1}",
            'T3': "Slot 3",
            'T4': f"Slot {solver.Value(T4) + 1}",
            'T5': f"Slot {solver.Value(T5) + 1}"
        }
        return schedule
    else:
        print('No solution found.')
        return None

def main():
    schedule = create_csp()
    if schedule:
        print("Final Valid Schedule:")
        for presentation, slot in schedule.items():
            print(f"{presentation} : {slot}")

if __name__ == '__main__':
    main()


TypeError: unsupported operand type(s) for |: 'ortools.sat.python.cp_model_helper.BoundedLinearExpression' and 'ortools.sat.python.cp_model_helper.BoundedLinearExpression'